# Heart Disease Logistic Regression — Amazon SageMaker Training & Testing

**Run this notebook inside a SageMaker notebook instance** (AWS Academy Learner Lab),
kernel `conda_python3`.

**Scope:** this notebook **trains and tests** the model only. No endpoint is created and no
model deployment service is used, in line with the AWS Academy account limitations for this course.

### Files that must be uploaded next to this notebook

```
sagemaker_heart_lr.ipynb      <- this notebook
train_heart_lr.py             <- training script (entry point)
data/heart_train.csv          <- preprocessed training split
data/heart_test.csv           <- preprocessed test split
data/preprocessing_config.json
```

The features in the CSVs are **already standardized** by the local notebook using the training-set
mean and standard deviation, and the hyperparameters are fixed. Together with zero initialization and
full-batch gradient descent, this makes the cloud run deterministic and directly comparable with the
local execution.



## 0. Environment

In [ ]:
import sagemaker, boto3, os, json, tarfile
import numpy as np
import pandas as pd

session = sagemaker.Session()
region = session.boto_region_name
bucket = session.default_bucket()
role = sagemaker.get_execution_role()
prefix = "heart-disease-lr"

print("sagemaker SDK :", sagemaker.__version__)
print("region        :", region)
print("default bucket:", bucket)
print("execution role:", role)

In [ ]:
# Sanity check: the required files must be present
for path in ["train_heart_lr.py",
             "data/heart_train.csv",
             "data/heart_test.csv",
             "data/preprocessing_config.json"]:
    print(("OK   " if os.path.exists(path) else "MISSING "), path)

config = json.load(open("data/preprocessing_config.json"))
print()
print("Hyperparameters:", {k: config[k] for k in ["alpha", "num_iters", "lambda", "threshold"]})
print("Local test metrics to reproduce:", config["local_test_metrics"])

## 1. Upload the preprocessed data to S3

In [ ]:
train_s3 = session.upload_data("data/heart_train.csv", bucket=bucket, key_prefix=f"{prefix}/train")
test_s3  = session.upload_data("data/heart_test.csv",  bucket=bucket, key_prefix=f"{prefix}/test")

print("train channel:", train_s3)
print("test  channel:", test_s3)

## 2. Define the training job

The `SKLearn` container is used in **script mode** purely as a managed Python runtime — the model
itself is the from-scratch NumPy implementation (sigmoid, binary cross-entropy, L2 penalty, batch
gradient descent) copied verbatim from the local notebook. `scikit-learn` is never called for
training.

In [ ]:
from sagemaker.sklearn.estimator import SKLearn

estimator = SKLearn(
    entry_point="train_heart_lr.py",
    source_dir=".",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    base_job_name="heart-disease-lr",
    hyperparameters={
        "alpha": config["alpha"],
        "num-iters": config["num_iters"],
        "lam": config["lambda"],
        "threshold": config["threshold"],
    },
)

print("Estimator configured.")
print("  entry point     :", estimator.entry_point)
print("  instance type   :", estimator.instance_type)
print("  hyperparameters :", estimator.hyperparameters())

## 3. Launch training

The cell below streams the training log. Capture a screenshot while it runs and another one when it prints `Completed`.

In [ ]:
estimator.fit({"train": train_s3, "test": test_s3}, logs=True)

job_name = estimator.latest_training_job.name
print("\nTraining job name:", job_name)

In [ ]:
# Evidence that training completed, read back from the SageMaker API
sm = boto3.client("sagemaker", region_name=region)
desc = sm.describe_training_job(TrainingJobName=job_name)

print("Training job   :", desc["TrainingJobName"])
print("Status         :", desc["TrainingJobStatus"])
print("Instance type  :", desc["ResourceConfig"]["InstanceType"])
print("Instance count :", desc["ResourceConfig"]["InstanceCount"])
print("Volume (GB)    :", desc["ResourceConfig"]["VolumeSizeInGB"])
print("Billable secs  :", desc.get("BillableTimeInSeconds"))
print("Training secs  :", desc.get("TrainingTimeInSeconds"))
print("Image          :", desc["AlgorithmSpecification"]["TrainingImage"].split("/")[-1])
print("Model artifact :", desc["ModelArtifacts"]["S3ModelArtifacts"])

## 4. Download the trained model artifact

In [ ]:
artifact = estimator.model_data
print("Downloading:", artifact)

os.makedirs("model", exist_ok=True)
session.download_data(path="model", bucket=bucket,
                      key_prefix="/".join(artifact.split("/")[3:]))

with tarfile.open("model/model.tar.gz") as tar:
    tar.extractall("model")

print("Extracted files:", sorted(os.listdir("model")))

In [ ]:
params = np.load("model/model.npz")
w = params["w"]
b = float(params["b"])

print("w =", np.round(w, 4))
print("b =", round(b, 4))
print("||w|| =", round(float(np.linalg.norm(w)), 4))

metrics_cloud = json.load(open("model/metrics.json"))
print("\nMetrics reported by the training job:")
print(json.dumps(metrics_cloud, indent=2))

## 5. Test the trained model on the held-out test set

The evaluation is done **inside this notebook**, on the test split, using the downloaded weights.
No endpoint and no inference service is involved — this is plain NumPy arithmetic on the artifact.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def predict(w, b, X, threshold=0.5):
    return (sigmoid(X @ w + b) >= threshold).astype(float)

def evaluate(y_true, y_pred):
    tp = float(np.sum((y_pred == 1) & (y_true == 1)))
    tn = float(np.sum((y_pred == 0) & (y_true == 0)))
    fp = float(np.sum((y_pred == 1) & (y_true == 0)))
    fn = float(np.sum((y_pred == 0) & (y_true == 1)))
    accuracy  = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1,
            "TP": tp, "TN": tn, "FP": fp, "FN": fn}


train_df = pd.read_csv("data/heart_train.csv")
test_df = pd.read_csv("data/heart_test.csv")

X_train = train_df.drop(columns="target").to_numpy(float)
y_train = train_df["target"].to_numpy(float)
X_test = test_df.drop(columns="target").to_numpy(float)
y_test = test_df["target"].to_numpy(float)

m_train = evaluate(y_train, predict(w, b, X_train))
m_test = evaluate(y_test, predict(w, b, X_test))

results = pd.DataFrame([
    {"set": "train", **{k: round(v, 4) for k, v in m_train.items()}},
    {"set": "test",  **{k: round(v, 4) for k, v in m_test.items()}},
])
print("SageMaker-trained model evaluated on the held-out test set")
results

In [ ]:
# Confusion matrix on the test set
cm = np.array([[m_test["TN"], m_test["FP"]],
               [m_test["FN"], m_test["TP"]]])

print("Confusion matrix (test set)")
print(pd.DataFrame(cm,
                   index=["actual: no disease", "actual: disease"],
                   columns=["pred: no disease", "pred: disease"]).astype(int))

## 6. Comparison: SageMaker vs local execution

In [ ]:
local = config["local_test_metrics"]

comparison = pd.DataFrame([
    {"environment": "Local (Jupyter, this laptop)",
     **{k: round(float(local[k]), 4) for k in ["accuracy", "precision", "recall", "f1"]}},
    {"environment": "Amazon SageMaker training job",
     **{k: round(m_test[k], 4) for k in ["accuracy", "precision", "recall", "f1"]}},
])
comparison["difference"] = (comparison.iloc[1][["accuracy", "precision", "recall", "f1"]].values -
                            comparison.iloc[0][["accuracy", "precision", "recall", "f1"]].values).sum()

print("Test-set metrics, local vs cloud")
comparison[["environment", "accuracy", "precision", "recall", "f1"]]

In [ ]:
max_diff = max(abs(m_test[k] - float(local[k])) for k in ["accuracy", "precision", "recall", "f1"])
print("Maximum absolute difference between local and SageMaker test metrics: {:.6f}".format(max_diff))
print("Identical results:" , max_diff < 1e-6)

### Reporting — SageMaker training and testing

**Process.** The already-preprocessed splits were uploaded to the SageMaker default S3 bucket as two
input channels (`train`, `test`). A SageMaker **training job** was launched on an `ml.m5.large`
instance running `train_heart_lr.py` in script mode; the script implements the same NumPy logistic
regression as the local notebook (sigmoid, binary cross-entropy with L2 penalty, batch gradient
descent) with `alpha=0.1`, `num_iters=20000`, `lambda=1.0`. On completion the job wrote `model.npz`
and `metrics.json` to `SM_MODEL_DIR`, which SageMaker packaged as `model.tar.gz` in S3. That artifact
was downloaded and the model was scored on the held-out test set inside this notebook.

**Evidence of completion.** `describe_training_job` reports status `Completed` together with the
instance configuration and billable time (printed in section 3).

**Differences from the local execution.** The test-set metrics match the local ones to within
floating-point tolerance. This is expected and is the point of the setup: the optimizer is
deterministic (zero initialization, full-batch gradient descent, fixed iteration count, no shuffling
and no random sampling), and normalization was applied *before* upload, so no statistics are
recomputed in the cloud. Any residual difference would come only from BLAS-level floating-point
ordering on a different CPU architecture, which is far below the fourth decimal place.

What *does* differ is everything around the model: the cloud run provisions a dedicated instance,
pulls a versioned container, and produces a named, timestamped, auditable training job with its
artifact stored in S3. For a model of this size that infrastructure costs more wall-clock time than
the training itself (seconds). The value is reproducibility and traceability, not speed — and it is
what makes the same pipeline viable when the dataset grows from 302 patients to a population-scale
registry.

**Endpoint deployment was intentionally not performed**, as it is outside the scope of the AWS
Academy accounts provided for this course.